In [1]:
%pip install -e /Users/instafiore/Workspace/AMOSUM

Obtaining file:///Users/instafiore/Workspace/AMOSUM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for amosum (pyproject.toml) ... done
  Created wheel for amosum: filename=amosum-0.0.1-py3-none-any.whl size=3315 sha256=8293723b3f2d872a8c9750dad08635397f7562c50fcb0000ded83e9dab574c61
  Stored in directory: /private/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/pip-ephem-wheel-cache-fp3uz03n/wheels/f9/7b/3a/a109cecea02e5123eee232e700087c069a0eb849a8a3ec42cb
Successfully built amosum
  Attempting uninstall: amosum
    Found existing installation: amosum 0.0.1
    Uninstalling amosum-0.0.1:
      Successfully uninstalled amosum-0.0.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
import argparse
import sys
from itertools import product
import os
import subprocess
from typing import Dict, List, Set
import re


from amosum.utility import *
from amosum.amoclingo.propagator_clingo_c.runner_clingo import *
from amosum.amoclingo.propagator_clingo_py.runner_clingo import *
from amosum.amowasp.propagator_wasp_py.runner_wasp import *
import sys
import sys
import os
import re

import os
import glob
import pathlib



In [3]:
class Checker:
    def __init__(self, args: Dict):
        solver = args.get("lang")
        language = args.get("lang")
        self.runner : RunnerClingoCpp | RunnerClingoPython
        if language == "py":
            if solver == "clingo":
                self.runner = RunnerClingoPython(parameters=args)
            else:
                self.runner = RunnerWasp(parameters=args)
        elif language == "cpp":
            self.runner = RunnerClingoCpp(parameters=args)
        self.runner = RunnerClingoCpp(args)
        self.args = args
        print(f"args: {args}")

    def check_correctness_all(self) -> bool:
        """
        Checks if each answer set produced by the propagator is also produced by clingo.

        Returns:
        bool: True if all propagator's answer sets are found in clingo's answer sets, False otherwise.
        """
        # Run the propagator
        self.runner.printOutput = False
        results: List[Result] = self.runner.run()
        def notAux(x: str):
            return re.search("^__", x) is None
        propagator_answer_sets = [set(filter(notAux, r.model.assigment)) for r in results] if results[0].exitCode != 20 else "UNSAT"
        if propagator_answer_sets == "UNSAT":
            print("Propagator found UNSAT")
            propagator_answer_sets = []


        # Run clingo
        clingo_answer_sets = run_clingo(self.args)
        if clingo_answer_sets == "UNSAT":
            print("Clingo found UNSAT")
            clingo_answer_sets = []

        correct = True 
        # Compare answer sets
        for pas in propagator_answer_sets:
            if pas not in clingo_answer_sets:
                print(f" => Answer set {pas} from propagator not found in clingo's answer sets.")
                correct = False
                break
        
        for pas in clingo_answer_sets:
            if pas not in propagator_answer_sets:
                print(f" <= Answer set {pas} from clingo not found in propagator's answer sets.")
                correct = False
                break

        if not correct:
            # debug("propagator_answer_sets: ")
            # for ans in propagator_answer_sets:
            #     debug(ans)
            # debug("clingo_answer_sets: ")
            # for ans in clingo_answer_sets:
            #     debug(ans)
            pass
        
        return correct
    
    def check_correctness_instance(self) -> bool:
        """
        Checks if an answer set produced by the propagator respects the bound.

        Returns:
        bool: True if the propagator's answer sets respects the bound, False otherwise.
        """

        encoding_path = self.args["encoding"]
        instance_path = self.args["instance"]
        checker_file = pathlib.Path(encoding_path).parent / "checker-amosum.asp"
    
        print(f"checker file: {checker_file}")
        results: List[Result] = self.runner.run()   
        result = results[0]

        reified_answerset = []
        if(result.exitCode == 20): return True

        for atom in result.model.assigment:
            reified_answerset.append(f"{atom}.")

        ctl = clingo.Control()
        ctl.load(str(checker_file))
        reified_answerset_str = "\n".join(reified_answerset)
        ctl.add(reified_answerset_str)
        ctl.ground()
        success: bool
        def on_finish(res: clingo.SolveResult):
            nonlocal success
            success = res.satisfiable
        ctl.solve(on_finish=on_finish)
        if success:
            print(f"check passed for: {self.args} ✅ ")
        else:
            print(f"check not passed for: {self.args} 😱")
        return success


def check():
    args = parse_args(checkCorrectness=True)

    c = args["check"]
    args["models"] = 0 if c == "all" else 1
    print(f"c: {c}")
    checker = Checker(args)
    print(f"[python] parameters: {args}")
    if c == "all":
        success = checker.check_correctness_all()
    elif c == "instance":
        success = checker.check_correctness_instance()

    return success

def test(benchmark: str, mode: str = "all", instances = "*"):
    encodings = [e for e in glob.glob(f"bench/{benchmark}/*") if re.search("encoding-amosum-amo", e)]
    instances = [e for e in glob.glob(f"bench/{benchmark}/instances/{instances}")]
    smpc = ["true", "false"]
    # smpc = ["true"]
    lazyness = ["true", "false", "hybrid"]
    print(f"encodings: {encodings}")
    success = True

    for e, i, s, l  in product(encodings, instances, smpc, lazyness):
        sys.argv = [sys.argv[0], f"-e={e}", f"-i={i}", f"-smpc={s}", f"-l={l}", mode]
        success = check()
        if not success:
            break
    print("[ALL] Check passed 🔥" if success else "[ALL] Check failed 🚨")

In [4]:
test(benchmark="kn_toy")

encodings: ['bench/kn_toy/encoding-amosum-amo.asp']
c: all
args: {'encoding': 'bench/kn_toy/encoding-amosum-amo.asp', 'instance': 'bench/kn_toy/instances/toy_le_4.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 0, 'log_file': 'log', 'static_mpc': 'true', 'check': 'all', 'solver': 'clingo'}
[python] parameters: {'encoding': 'bench/kn_toy/encoding-amosum-amo.asp', 'instance': 'bench/kn_toy/instances/toy_le_4.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 0, 'log_file': 'log', 'static_mpc': 'true', 'check': 'all', 'solver': 'clingo'}
encoding: bench/kn_toy/encoding-amosum-amo.asp
instance: bench/kn_toy/instances/toy_le_4.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-amo_without_amosum_2026-08-04-11-44-21-564142.asp            -instance=/tmp/.toy_le_4_without_amosum_2026-08-04-11-44-21-606337.asp             -models=0             -logfile=log             -ser

In [5]:
test(benchmark="kn", mode="instance")

encodings: ['bench/kn/encoding-amosum-amo.asp']
c: instance
args: {'encoding': 'bench/kn/encoding-amosum-amo.asp', 'instance': 'bench/kn/instances/0005-knapsack-5-12264-254036-type2.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
[python] parameters: {'encoding': 'bench/kn/encoding-amosum-amo.asp', 'instance': 'bench/kn/instances/0005-knapsack-5-12264-254036-type2.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
checker file: bench/kn/checker-amosum.asp
encoding: bench/kn/encoding-amosum-amo.asp
instance: bench/kn/instances/0005-knapsack-5-12264-254036-type2.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-amo_without_amosum_2026-08-04-11-44-25-843471.asp            -instance=/tmp/.000

In [6]:
test(benchmark="ga", mode="instance")

encodings: ['bench/ga/encoding-amosum-amo.asp']
c: instance
args: {'encoding': 'bench/ga/encoding-amosum-amo.asp', 'instance': 'bench/ga/instances/000-group-assignment-30-10-sat.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
[python] parameters: {'encoding': 'bench/ga/encoding-amosum-amo.asp', 'instance': 'bench/ga/instances/000-group-assignment-30-10-sat.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
checker file: bench/ga/checker-amosum.asp
encoding: bench/ga/encoding-amosum-amo.asp
instance: bench/ga/instances/000-group-assignment-30-10-sat.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-amo_without_amosum_2026-08-04-11-44-31-780021.asp            -instance=/tmp/.000-group-assig

In [7]:
test(benchmark="gc", mode="instance")

encodings: ['bench/gc/encoding-amosum-amo.asp']
c: instance
args: {'encoding': 'bench/gc/encoding-amosum-amo.asp', 'instance': 'bench/gc/instances/0001-graph_colouring-125-0_1200.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
[python] parameters: {'encoding': 'bench/gc/encoding-amosum-amo.asp', 'instance': 'bench/gc/instances/0001-graph_colouring-125-0_1200.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
checker file: bench/gc/checker-amosum.asp
encoding: bench/gc/encoding-amosum-amo.asp
instance: bench/gc/instances/0001-graph_colouring-125-0_1200.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-amo_without_amosum_2026-08-04-11-44-44-338054.asp            -instance=/tmp/.0001-graph_c

In [8]:
test(benchmark="nr", mode="instance")

encodings: ['bench/nr/encoding-amosum-amo.asp']
c: instance
args: {'encoding': 'bench/nr/encoding-amosum-amo.asp', 'instance': 'bench/nr/instances/10nurses.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
[python] parameters: {'encoding': 'bench/nr/encoding-amosum-amo.asp', 'instance': 'bench/nr/instances/10nurses.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
checker file: bench/nr/checker-amosum.asp
encoding: bench/nr/encoding-amosum-amo.asp
instance: bench/nr/instances/10nurses.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-amo_without_amosum_2026-08-04-11-44-45-601162.asp            -instance=/tmp/.10nurses_without_amosum_2026-08-04-11-44-45-638582.asp             -models=1     

In [9]:
test(benchmark="ga", mode="instance", instances="097-group-assignment-50-20-middle.asp")

encodings: ['bench/ga/encoding-amosum-amo.asp']
c: instance
args: {'encoding': 'bench/ga/encoding-amosum-amo.asp', 'instance': 'bench/ga/instances/097-group-assignment-50-20-middle.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
[python] parameters: {'encoding': 'bench/ga/encoding-amosum-amo.asp', 'instance': 'bench/ga/instances/097-group-assignment-50-20-middle.asp', 'lazy': 'true', 'reason': 'nomin', 'lang': 'cpp', 'models': 1, 'log_file': 'log', 'static_mpc': 'true', 'check': 'instance', 'file_answerset': None}
checker file: bench/ga/checker-amosum.asp
encoding: bench/ga/encoding-amosum-amo.asp
instance: bench/ga/instances/097-group-assignment-50-20-middle.asp
run:	 /Users/instafiore/Workspace/AMOSUM/amosum/amoclingo/propagator_clingo_c/bin/./amosum_cpp            -encoding=/tmp/.encoding-amosum-amo_without_amosum_2026-08-04-11-45-48-903110.asp            -instance=/tmp/.097-gr